# 14장 (3단계) - 웹 검색 + RAG 도구 (tools)\n\nTavily 웹 검색 → JSON 저장 → 랭체인 `Document` 변환 → 청킹 → Chroma 벡터 DB 저장, 그리고 벡터 검색(`retrieve`).\n\n> `.env`에 `OPENAI_API_KEY`(OpenRouter), `TAVILY_API_KEY`가 필요합니다. 노트북은 **chap14 폴더에서** 실행하세요.

## 설정 (임베딩 · Chroma 벡터스토어)

In [1]:
from tavily import TavilyClient
from langchain_core.tools import tool
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from datetime import datetime
from dotenv import load_dotenv
import json
import os

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# 노트북은 __file__ 이 없으므로 현재 작업 폴더(chap14)를 기준 경로로 사용
current_path = os.getcwd()
output_dir = os.path.join(current_path, 'data')   # 검색 결과(JSON) 저장 폴더
os.makedirs(output_dir, exist_ok=True)

# 오픈AI Embedding 설정 (OpenAI API 대신 OpenRouter 사용)
embedding = OpenAIEmbeddings(
    model='text-embedding-3-large',
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENAI_API_KEY,
)

# 크로마 DB 저장 경로 설정
persist_directory = f"{current_path}/data/chroma_store"

vectorstore = Chroma(
    persist_directory=persist_directory,
    embedding_function=embedding,
)

/Users/seminy/Desktop/Main/Git/AI_Bootcamp/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
USER_AGENT environment variable not set, consider setting it to identify your requests.
/Users/seminy/Desktop/Main/Git/AI_Bootcamp/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 웹 검색 (`web_search`) 과 페이지 로더 (`load_web_page`)

In [2]:
@tool
def web_search(query: str):
    """
    주어진 query에 대해 웹검색을 하고, 결과를 반환한다.

    Args:
        query (str): 검색어

    Returns:
        list: 검색 결과
    """
    client = TavilyClient()

    content = client.search(
        query,
        search_depth="advanced",
        include_raw_content=True,
    )

    results = content["results"]

    for result in results:
        if result["raw_content"] is None:
            try:
                result["raw_content"] = load_web_page(result["url"])
            except Exception as e:
                print(f"Error loading page: {result['url']}")
                print(e)
                result["raw_content"] = result["content"]

    # 검색 결과를 JSON 파일로 저장
    file_name = f"resources_{datetime.now().strftime('%Y_%m%d_%H%M%S')}.json"
    resources_json_path = os.path.join(output_dir, file_name)

    with open(resources_json_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=4)

    return results, resources_json_path

In [3]:
def load_web_page(url: str):
    """WebBaseLoader로 URL의 본문을 읽어와 공백을 정리한 텍스트로 반환한다."""
    loader = WebBaseLoader(url, verify_ssl=False)

    content = loader.load()
    raw_content = content[0].page_content.strip()

    while '\n\n\n' in raw_content or '\t\t\t' in raw_content:
        raw_content = raw_content.replace('\n\n\n', '\n\n')
        raw_content = raw_content.replace('\t\t\t', '\t\t')

    return raw_content

## 검색 결과 → Document 변환

In [4]:
def web_page_to_document(web_page):
    """검색 결과(dict) 하나를 랭체인 Document 객체로 변환한다."""
    if len(web_page['raw_content']) > len(web_page['content']):
        page_content = web_page['raw_content']
    else:
        page_content = web_page['content']

    document = Document(
        page_content=page_content,
        metadata={
            'title': web_page['title'],
            'source': web_page['url'],
        }
    )

    return document


def web_page_json_to_documents(json_file):
    """JSON 파일에 저장된 검색 결과들을 Document 객체 리스트로 변환한다."""
    with open(json_file, "r", encoding='utf-8') as f:
        resources = json.load(f)

    documents = []

    for web_page in resources:
        document = web_page_to_document(web_page)
        documents.append(document)

    return documents

## 청킹 · Chroma 저장

In [5]:
def split_documents(documents, chunk_size=1000, chunk_overlap=100):
    """Document 객체들을 청크 단위로 자른다."""
    print('Splitting documents...')
    print(f"{len(documents)}개의 문서를 {chunk_size}자 크기로 중첩 {chunk_overlap}자로 분할합니다.\n")

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap
    )

    splits = text_splitter.split_documents(documents)

    print(f"총 {len(splits)}개의 문서로 분할되었습니다.")
    return splits

In [6]:
def documents_to_chroma(documents, chunk_size=1000, chunk_overlap=100):
    """새로운 Document들만 청킹해 Chroma DB에 저장한다(중복 URL은 건너뜀)."""
    print("Documents를 Chroma DB에 저장합니다.")
    urls = [document.metadata['source'] for document in documents]
    stored_metadatas = vectorstore._collection.get()['metadatas']
    stored_web_urls = [metadata['source'] for metadata in stored_metadatas]
    new_urls = set(urls) - set(stored_web_urls)
    new_documents = []

    for document in documents:
        if document.metadata['source'] in new_urls:
            new_documents.append(document)
            print(document.metadata)

    splits = split_documents(new_documents, chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    if splits:
        vectorstore.add_documents(splits)
    else:
        print("No new urls to process")


def add_web_pages_json_to_chroma(json_file, chunk_size=1000, chunk_overlap=100):
    """json 파일에서 documents를 만들고, 그 documents들을 Chroma DB에 저장한다."""
    documents = web_page_json_to_documents(json_file)
    documents_to_chroma(
        documents,
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )

## 벡터 검색 (`retrieve`)

In [7]:
@tool
def retrieve(query: str, top_k: int = 5):
    """
    주어진 query에 대해 벡터 검색을 수행하고, 결과를 반환한다.
    """
    retriever = vectorstore.as_retriever(search_kwargs={"k": top_k})
    retrieved_docs = retriever.invoke(query)

    return retrieved_docs

## 실행 예시\n웹 검색 → 벡터 DB 저장 → 검색까지 한 번에 실행합니다.

In [10]:
# 1) 웹 검색 → JSON 저장
results, resources_json_path = web_search.invoke({"query": "2025년 한국 경제 전망"})
print("검색 결과 저장 경로:", resources_json_path)

# 2) JSON → Document 변환 → 청킹 → Chroma DB에 저장
add_web_pages_json_to_chroma(resources_json_path)

# 3) 벡터 검색으로 확인
retrieved_docs = retrieve.invoke({"query": "한국 경제 위험 요소"})
for doc in retrieved_docs:
    print(doc.metadata)
    print(doc.page_content[:200])
    print('---')

검색 결과 저장 경로: /Users/seminy/Desktop/Main/Git/AI_Bootcamp/chap14/data/resources_2026_0706_135436.json
Documents를 Chroma DB에 저장합니다.
Splitting documents...
0개의 문서를 1000자 크기로 중첩 100자로 분할합니다.

총 0개의 문서로 분할되었습니다.
No new urls to process
{'title': '[PDF] 2025년 한국 경제 전망(수정)', 'source': 'https://hri.co.kr/upload/board/2887055909_vDf0PB9V_20250507061207.pdf'}
< 현대경제연구원의 2025년 한국 경제 전망표 (2025년 4월) > 구 분 2023 2024 2025(E) 상반 하반 연간 상반 하반 연간 국 민 계 정 경제성장률 (%) 1.4 2.8 1.3 2.0 0.4 0.9 0.7 민간소비 (%) 1.8 1.0 1.3 1.1 0.6 1.2 0.9 건설투자 (%) 1.5 0.4 △6.1 △3.0 △10.0 △2.0
---
{'source': 'https://hri.co.kr/upload/board/2887055909_vDf0PB9V_20250507061207.pdf', 'title': '[PDF] 2025년 한국 경제 전망(수정)'}
World Bank Commodity Price Data를 이 용한 지수화.
자료: IMF WEO(April 2025).
2025년 한국 경제 전망(수정) 8 2. 최근 한국 경제 경기 동향 ① 경기(경제성장률) ¡ (역성장 현실화) 한국 경제는 1분기 현재 경기지수 상 반등하는 모습이나, 내 수와 수출이 모두 크게 침체되면서 경기 회복의 가능성은 희박한 
---
{'source': 'https://hri.co.kr/upload/board/2887055909_vDf0PB9V_20250507061207.pdf', 'title': '[PDF] 2025년 한국 경제 전망(수정)'